```
# Lab type:  prompt
# Course:    NL301 Natural Language Processing with Python
# Lesson:    06 — LLM-Assisted NLP
# Task:      Complete three prompt templates for support-ticket classification
#            using the Anthropic API.
```

## Setup

Requires an `ANTHROPIC_API_KEY`:

1. Go to <a href="https://console.anthropic.com/settings/keys" target="_blank" rel="noopener noreferrer">console.anthropic.com</a> and sign in.
2. Click **Create Key** and copy it.
3. **In Colab:** open the Secrets panel (🔑 icon in the left sidebar), add a secret named `ANTHROPIC_API_KEY`, and paste your key.
   **Locally:** set it in your shell before launching Jupyter: `export ANTHROPIC_API_KEY="your-key"`.

In [ ]:
!pip install anthropic --quiet

In [ ]:
import os

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("Anthropic API key loaded from Colab secrets.")
except ImportError:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
        if ANTHROPIC_API_KEY:
            print("Anthropic API key loaded from .env file.")
        else:
            print("Anthropic API key not found in .env file. Please ensure ANTHROPIC_API_KEY is set.")
    except ImportError:
        print("python-dotenv not installed. Please install it (`pip install python-dotenv`) or ensure ANTHROPIC_API_KEY is set as an environment variable.")
        ANTHROPIC_API_KEY = None

if ANTHROPIC_API_KEY is None:
    raise ValueError("ANTHROPIC_API_KEY is not set. Please set it in Colab secrets or a .env file.")

import anthropic
import json
from typing import Optional

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Support ticket dataset
tickets = [
    {"id": 1, "text": "My order arrived damaged and I want a full refund immediately."},
    {"id": 2, "text": "The product exceeded my expectations — absolutely fantastic!"},
    {"id": 3, "text": "Can you tell me whether you ship to Canada?"},
    {"id": 4, "text": "I've been waiting three weeks and still no delivery."},
    {"id": 5, "text": "Do you offer student discounts on annual subscriptions?"},
    {"id": 6, "text": "The app keeps crashing on iOS 17. This is unacceptable."},
    {"id": 7, "text": "Just wanted to say your support team was incredibly helpful."},
    {"id": 8, "text": "Please cancel my subscription effective immediately."},
]
CATEGORIES = ["complaint", "praise", "inquiry"]

---
## Task 1: Zero-shot classification

Complete the `zero_shot_prompt` so that the model returns exactly one word: `complaint`, `praise`, or `inquiry`. No other output.

In [ ]:
def classify_zero_shot(text: str) -> str:
    # TODO: write a prompt that returns exactly one category word.
    # Hint: be explicit about output format — one word, nothing else.
    zero_shot_prompt = f"""
    YOUR PROMPT HERE
    
    Ticket: {text}
    """
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=16,
        messages=[{"role": "user", "content": zero_shot_prompt}],
    )
    return response.content[0].text.strip().lower()

# Test on first 4 tickets
for t in tickets[:4]:
    label = classify_zero_shot(t["text"])
    print(f"[{label:10s}]  {t['text'][:60]}")

**Analysis:** Does zero-shot correctly classify the complaint about a damaged order? The cancellation request — is that a complaint or inquiry?

*(Write your answer here.)*

---
## Task 2: Few-shot classification

Add 2 labelled examples per category to your prompt. Then compare zero-shot vs few-shot accuracy on all 8 tickets (use manual ground-truth labels below).

In [ ]:
GROUND_TRUTH = {1: "complaint", 2: "praise", 3: "inquiry", 4: "complaint",
                5: "inquiry",   6: "complaint", 7: "praise", 8: "inquiry"}

def classify_few_shot(text: str) -> str:
    # TODO: build a few-shot prompt with 2 examples per category (6 examples total).
    # Each example: show the ticket text and the correct label.
    few_shot_prompt = f"""
    YOUR FEW-SHOT PROMPT HERE
    
    Ticket: {text}
    """
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=16,
        messages=[{"role": "user", "content": few_shot_prompt}],
    )
    return response.content[0].text.strip().lower()

# Evaluate both approaches
def accuracy(fn):
    correct = sum(fn(t["text"]) == GROUND_TRUTH[t["id"]] for t in tickets)
    return correct / len(tickets)

print(f"Zero-shot accuracy: {accuracy(classify_zero_shot):.2f}")
print(f"Few-shot  accuracy: {accuracy(classify_few_shot):.2f}")

**Analysis:** Which approach performed better? When would you expect few-shot to consistently outperform zero-shot?

*(Write your answer here.)*

---
## Task 3: Structured JSON extraction

Write a prompt that extracts four fields from each ticket: `sentiment` (positive/negative/neutral), `issue_type` (billing/shipping/technical/general), `urgency` (low/medium/high), `action_required` (bool). Handle `json.JSONDecodeError`.

In [ ]:
from dataclasses import dataclass

@dataclass
class TicketAnalysis:
    sentiment: str
    issue_type: str
    urgency: str
    action_required: bool

def extract_ticket_info(text: str) -> Optional[TicketAnalysis]:
    # TODO: write a prompt instructing the model to return ONLY valid JSON
    # with exactly these four keys.  Wrap json.loads in a try/except.
    extraction_prompt = f"""
    YOUR EXTRACTION PROMPT HERE
    
    Ticket: {text}
    """
    
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": extraction_prompt}],
        )
        data = json.loads(response.content[0].text.strip())
        return TicketAnalysis(**data)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        print(f"Raw response: {response.content[0].text[:200]}")
        return None

for t in tickets[:3]:
    result = extract_ticket_info(t["text"])
    print(f"Ticket {t['id']}: {result}")

**Analysis questions:**

1. At roughly 200 input tokens + 50 output tokens per ticket, and Claude Haiku pricing (~$0.80 per 1M input tokens, ~$4 per 1M output tokens), what is the approximate daily API cost for 10,000 tickets?
2. Under what conditions would few-shot reliably outperform zero-shot for classification tasks like this?
3. In a production pipeline, beyond `JSONDecodeError`, what other error scenarios should you handle?

*(Write your answers here.)*